# Historical notebook — source archive

Original: `3_10_Ved (1).ipynb`. Credentials and cloud endpoint removed; outputs cleared. This is provenance, **not a supported Run All workflow**. See the four notebooks above this archive for the maintained pipeline. Historical errors, global state, and incompatible APIs may remain. Cloud collection-reset cells have been disabled.


In [ ]:
import sqlite3
import os
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch

# Database setup
DB_PATH = "mllm_experiments.db"

def create_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS experiment_results4 (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            image_name TEXT,
            question TEXT,
            context TEXT,
            object TEXT,
            grounding TEXT,
            image_tag TEXT,
            date TEXT,
            model_response TEXT
        )
    """)
    conn.commit()
    conn.close()

create_db()



def store_experiment_result(image_name, tags, response):
    """Safe database insertion with all required fields"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO experiment_results4
        (image_name, question, context, object, grounding, image_tag, date, model_response)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        image_name,
        tags["question"],
        tags["context"],
        tags["object"],
        tags["grounding"],
        tags["image"],
        tags.get("date", "2023-01-01"),
        response
    ))

    conn.commit()
    conn.close()
    print(f"Stored result for {image_name}")

In [ ]:
import re
def clean_response(response):
    """Clean the response by removing unnecessary tags and repetitive text."""
    # Remove unwanted tags and noise
    unwanted_patterns = [
        r"<image>.*?</image>",  # Remove <image> tags and their content
        r"<grounding>.*?</grounding>",  # Remove <grounding> tags
        r"<context>.*?</context>",  # Remove <context> tags
        r"<image>.*?</image>",  # Remove <image> tags
        r"<object>.*?</object>",  # Remove <object> tags
        r"What can you see in this Sentinel-2A satellite image of.*?\?",  # Remove repetitive question
    ]

    for pattern in unwanted_patterns:
        response = re.sub(pattern, "", response, flags=re.DOTALL)

    # Remove extra spaces and newlines
    response = " ".join(response.split())

    return response.strip()


Formalized Tags

In [ ]:
import torch
torch.cuda.empty_cache()

In [ ]:
#ran over 10 images in the drive
import os
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch
import pandas as pd


# Disable oneDNN optimizations if needed
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# Set the path where your data is stored
datapath = r"/content"  # <-- Update this path to where your images are stored

# Load the Kosmos-2 model (this will download ~7GB on first run)
model_source = "microsoft/kosmos-2-patch14-224"
processor = AutoProcessor.from_pretrained(model_source)
model = AutoModelForVision2Seq.from_pretrained(model_source)

if torch.cuda.is_available():
    model = model.to("cuda")
    print("Using CUDA")

# ✅ Generate prompts for multiple images
# ✅ Generate prompts for multiple images
def generate_prompt(tags):

    """
    Generates a structured paratactic prompt.

    :param question: The main question.
    :param tags: Keyword arguments representing tag-value pairs.
    :return: Formatted prompt string.
    """
    prompt = f"""<grounding> {tags['grounding']} </grounding>
    <image> {tags['image']} </image>
    <context> {tags['context']} </context>
    <object> {tags['object']} </object>
    {tags['question']}"""
    return prompt

# ✅ List of input images to process
image_files = [
    # "LA-GettyMuseum_openEO-2025-01-12Z_34_118_1738974467.png",
    # "Damascus_openEO_2025-01-14Z_33_36_1738966392.png",
    "ndvi_ndvi_Paris_openEO_2025-02-01Z_anthropic.png",
    # "Canberra_openEO_2025-01-13Z_35_149_1738976955.png",
    # "fmr_fmr_openEO_2025-01-24Z_chatgpt_copper.png",
    # "ndvi_ndvi_Paris_openEO_2025-02-01Z_chatgpt.png",
    # "openEO_2024-12-30Z_36_111_1738893776.png",
    # "openEO_2025-01-06Z_37_114_1738892548.png",
    # "openEO_2024-12-31Z_34_118_1738890849.png",
    # # "openEO_2025-01-29Z_25_82_1738892257.png",
    # "openEO_2025-01-30Z_51_0_1738890156.png",
    # # "openEO_2024-12-31Z_32_117_1738891932.png"

]

def metadata(filename):
  parts = filename.split("_")
  if len(parts)>0:
    location = parts[0]
  else:
    location = ""

  for i in parts:
    if len(i) == 10 and i[4] == "-" and i[7] == "-":
      date = i
      break
    else:
      date = ""

  if "fmr" in filename.lower():
    additional_info = "Ferrous Mineral Ratio (FMR)"
  elif "ndvi" in filename.lower():
    additional_info = "Normalized Difference Vegetation Index (NDVI)"
  else:
    additional_info = "Unknown"

  return {"location": location, "date": date, "additional_info": additional_info}



for img_file in image_files:
    try:
        # Extract metadata
        meta_data = metadata(img_file)
        location = meta_data["location"]
        date = meta_data["date"]
        info = meta_data["additional_info"]

        # Generate question

        question = f"What can you see in this Sentinel-2A satellite image of {location}?"

        # Prepare tags
        tags = {
            "question": question,
            "grounding": f"Sentinel-2A satellite data.",
            "context": f"This is a satellite image derived from Sentinel-2A of {location} captured on {date}.",
            "image": f"Analyze the {info} in the image.",
            "object": f"Identify key objects in the image related to {info}.",
            "date": date
        }


        # Generate prompt
        prompt = generate_prompt(tags)


        # Process image
        image = Image.open(os.path.join(datapath, img_file))
        inputs = processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding=True
        )

        if torch.cuda.is_available():
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                min_length=8,
                num_beams=5,
                max_new_tokens=128,
                length_penalty=1.0
            )

        response = processor.batch_decode(outputs, skip_special_tokens=True)[0]

        # Clean the response
        cleaned_response = clean_response(response)

        # Store results
        store_experiment_result(img_file, tags, cleaned_response)
        print(f"Processed {img_file}\nResponse: {cleaned_response}\n{'='*80}")

    except Exception as e:
        print(f"Error processing {img_file}: {str(e)}")

def export_to_csv():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query("SELECT * FROM experiment_results4", conn)
    df.to_csv("experiment_results4.csv", index=False)
    conn.close()
    print("Data exported to experiment_results4.csv")

export_to_csv()




In [ ]:
import os
import re
import sqlite3
import pandas as pd
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch

# Disable oneDNN optimizations if needed
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

# Set the path where your data is stored
datapath = r"/content"  # <-- Update this path to where your images are stored

# Load the Kosmos-2 model (this will download ~7GB on first run)
model_source = "microsoft/kosmos-2-patch14-224"
processor = AutoProcessor.from_pretrained(model_source)
model = AutoModelForVision2Seq.from_pretrained(model_source)

if torch.cuda.is_available():
    model = model.to("cuda")
    print("Using CUDA")

# Database setup
DB_PATH = "experiment_results.db"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS experiment_results3 (
        filename TEXT,
        location TEXT,
        focus TEXT,
        date TEXT,
        response TEXT
    )
''')
conn.commit()

# ✅ Generate prompts for multiple images
def generate_prompt(question, tags):
    return f"""
    <grounding>{tags['grounding']}</grounding>
    <context>{tags['context']}</context>
    <image>{tags['image']}</image>
    <object>{tags['object']}</object>
    {question}
    """

# ✅ Extract metadata from filename
def extract_metadata(filename):
    """
    Extracts location, focus, and date from the filename.
    """
    # Example filename: "LA-GettyMuseum_openEO-2025-01-12Z_34_118_1738974467.png"
    match = re.search(r"(.+)_openEO_(\d{4}-\d{2}-\d{2})Z_.+\.png", filename)
    if match:
        location = match.group(1)  # Extracts "LA-GettyMuseum"
        date = match.group(2)  # Extracts "2025-01-12"
    else:
        location = "Unknown"
        date = "Unknown"

    # Determine focus based on filename
    if "fmr" in filename.lower():
        focus = "Ferrous Minerals Ratio (FMR)"
    elif "ndvi" in filename.lower():
        focus = "Normalized Difference Vegetation Index (NDVI)"
    else:
        focus = "Unknown"

    return {"location": location, "date": date, "focus": focus}

# ✅ Clean the response
def clean_response(response):
    """
    Cleans the response text by removing unwanted tokens or formatting.
    """
    return response.strip()

# ✅ Store results in the database
def store_experiment_result(filename, tags, response):
    cursor.execute('''
        INSERT INTO experiment_results3 (filename, location, focus, date, response)
        VALUES (?, ?, ?, ?, ?)
    ''', (filename, tags["context"], tags["focus"], tags["date"], response))
    conn.commit()

# ✅ List of input images to process
image_files = [
#     "LA-GettyMuseum_openEO-2025-01-12Z_34_118_1738974467.png",
#     "Damascus_openEO_2025-01-14Z_33_36_1738966392.png",
    "ndvi_ndvi_Paris_openEO_2025-02-01Z_anthropic.png"
#     "Canberra_openEO_2025-01-13Z_35_149_1738976955.png",
#     "fmr_fmr_openEO_2025-01-24Z_chatgpt_copper.png",
#     "openEO_2024-12-30Z_36_111_1738893776.png",
#     "openEO_2025-01-06Z_37_114_1738892548.png",
#     "openEO_2024-12-31Z_34_118_1738890849.png",
#     "openEO_2025-01-29Z_25_82_1738892257.png",
#     "openEO_2024-12-31Z_32_117_1738891932.png"
# ]
]

# ✅ Process each image
for img_file in image_files:
    try:
        # Extract metadata from filename
        metadata = extract_metadata(img_file)
        location = metadata["location"]
        date = metadata["date"]
        focus = metadata["focus"]

        # Generate question
        question = f"What can you see in this Sentinel-2A satellite image of {location}?"

        # Prepare tags
        tags = {
            "question": question,
            "context": f"Location: {location}. The image is derived from Sentinel-2A satellite data and highlights {focus}. Date: {date}.",
            "object": "Identify key objects related to the feature.",
            "grounding": f"Sentinel-2A satellite data ({focus}).",
            "image": f"Analyze {focus} distribution. Bright regions indicate high concentrations, while darker areas represent lower values.",
            "focus": focus,
            "gps": "N/A",  # Replace with actual GPS coordinates if available
            "date": date,  # Extracted from filename
            "resolution": "10m"  # Replace with actual resolution if available
        }

        # Generate prompt
        prompt = generate_prompt(question, tags)

        # Process image
        image = Image.open(os.path.join(datapath, img_file))
        inputs = processor(
            text=prompt,
            images=image,
            return_tensors="pt",
            padding=True
        )

        if torch.cuda.is_available():
            inputs = {k: v.to("cuda") for k, v in inputs.items()}

        # Generate response
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                min_length=8,
                num_beams=5,
                max_new_tokens=128,
                length_penalty=1.0
            )

        response = processor.batch_decode(outputs, skip_special_tokens=True)[0]

        # Clean and store response
        cleaned_response = clean_response(response)
        store_experiment_result(img_file, tags, cleaned_response)
        print(f"Processed {img_file}\nResponse: {cleaned_response}\n{'='*80}")

    except Exception as e:
        print(f"Error processing {img_file}: {str(e)}")

# ✅ Export results to CSV
def export_to_csv():
    df = pd.read_sql_query("SELECT * FROM experiment_results3", conn)
    df.to_csv("experiment_results3.csv", index=False)
    print("Data exported to experiment_results3.csv")

export_to_csv()

# Close the database connection
conn.close()

RUN #1 Grounding has source, Context has real time data, image has the focus

Vectorizing (Vinaya's approach)

Applying vectorizations to all images tested above

In [ ]:
!pip install faiss-cpu


In [ ]:
# --- Step 1: Vectorize the Input Image and Candidate Output Texts using CLIP ---

from transformers import CLIPProcessor, CLIPModel
import torch
from PIL import Image
import numpy as np
import faiss

# Load the CLIP model and processor (using the base CLIP model)
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Load your input image (update the path as needed)
datapath = "/content"
image_files = [
    "Damascus_openEO_2025-01-14Z_33_36_1738966392.png",
    "ndvi_ndvi_Paris_openEO_2025-02-01Z_anthropic.png",
    "fmr_fmr_openEO_2025-01-24Z_chatgpt_copper.png",
    "ndvi_ndvi_Paris_openEO_2025-02-01Z_chatgpt.png",
    "openEO_2024-12-30Z_36_111_1738893776.png",
    "openEO_2025-01-06Z_37_114_1738892548.png",
    "openEO_2024-12-31Z_34_118_1738890849.png",
    "openEO_2025-01-29Z_25_82_1738892257.png",
    "openEO_2025-01-30Z_51_0_1738890156.png",
    "openEO_2024-12-31Z_32_117_1738891932.png"
]
images = [Image.open(os.path.join(datapath, img)) for img in image_files]

# Your candidate MLLM output texts (five variants)
output_texts = [
  " The image features an aerial view of a large city, likely in a developing country. The city sprawls across the landscape, covering a significant portion of the image's area. There are numerous cars and trucks scattered throughout the city, with some parked close together and others spread out. In addition to the vehicles, there are a few people visible in the scene. Some of them are standing or walking around, while others appear to be engaged in various activities. Overall, the city appears to be bustling with activity, with people going about their daily lives.",
  " In the image, you can see a variety of objects scattered across the scene. Some of these objects are green, while others are white. The green objects include trees, bushes, and grass. The white objects include buildings, roads, and other structures. The presence of these green and white objects suggests that the image was taken during a time when vegetation was green and the buildings and roads were white.",
  " In the image, you can see a dark area with a city in the middle of it. The city appears to be located in a mountainous region. There is also<phrase> a lake</phrase><object><patch_index_0790><patch_index_1023></object> in the area, which is located near the center of the city. Additionally, there is a mountain range in the background, which adds to the overall atmosphere of the scene.",
" The image shows a map of Paris, France, with a red line running through the center of the city.",
" In the image, you can see a satellite image of the city of Bamako, Mali. The city is located in the center of the image and is surrounded by a river. There are numerous buildings scattered throughout the city, some of which are close to the river, while others are further away. Some of the buildings are located near the river and others are located further away from the river. In addition to the buildings, there are a few cars and trucks visible in the scene, indicating that the city is bustling with activity.",
" The image features an aerial view of a city, likely in the United States. The city appears to be densely populated, with numerous buildings spread across the landscape. Some of the buildings are located near the center of the image, while others are scattered throughout the area. A river can be seen running through the city, adding to the urban atmosphere. Additionally, there are a few cars visible in the scene, likely belonging to the residents of the city. Overall, the image portrays a bustling city with a mix of residential and commercial structures.",
" In the image, you can see an aerial view of a city with a river running through it. There are several buildings in the city, some of which are located near the river, while others are located further away. The city appears to have a mix of residential and commercial areas, with some buildings located close to the river and others located farther away. Additionally, there are a few cars visible in the scene, which might belong to residents or visitors to the city. Overall, the image captures a snapshot of the city's urban landscape.",
" In the image, you can see a large city with a river running through it. The city appears to be located near a river, and there are numerous buildings scattered throughout the area. Some of these buildings are located close to the river, while others are further away. Additionally, there are trees in the area, adding to the urban landscape.",
"The image shows a satellite image of a city with a river running through it. The river is surrounded by a densely populated urban area. There are numerous buildings scattered throughout the city, with some located close to the river and others further away. Some of the buildings are located near the river, while others are located in different areas of the city. The city appears to be a mix of residential, commercial, and industrial areas.",
" In the image, you can see a satellite image of a city with buildings, roads, and trees. The city appears to be located in a rural area, and there are numerous buildings scattered throughout the area. Some of the buildings are located near the center of the city, while others are located towards the edges of the scene. There are also trees in the area, with some located close to the buildings and others further away. Overall, the image depicts a densely populated urban area with a mix of residential and commercial structures.",

]

# Process the image to obtain its embedding
metadata = {}

# ✅ FAISS index initialization
D = 512  # CLIP embedding size
index = faiss.IndexFlatL2(D)

# ✅ Process all images + captions
for i, img_name in enumerate(image_files):
    image = images[i]
    caption = output_texts[i]


    # --- Vectorize Image ---
    img_inputs = clip_processor(images=image, return_tensors="pt")
    with torch.no_grad():
        img_embedding = clip_model.get_image_features(**img_inputs)
    img_embedding = img_embedding / img_embedding.norm(p=2, dim=-1, keepdim=True)
    img_vec = img_embedding.cpu().numpy()

    # --- Vectorize Caption ---
    text_inputs = clip_processor(text=caption, return_tensors="pt", truncation=True, max_length=77)
    with torch.no_grad():
        text_embedding = clip_model.get_text_features(**text_inputs)
    text_embedding = text_embedding / text_embedding.norm(p=2, dim=-1, keepdim=True)
    text_vec = text_embedding.cpu().numpy()

    # --- Add to FAISS ---
    index.add(text_vec)  # Only storing text embeddings
    metadata[i] = {"image": img_name, "caption": caption}  # Map FAISS index to metadata

print(f"✅ Stored {index.ntotal} vectors in FAISS")

# --- Step 2: Querying the Vector Database ---

# ✅ Query FAISS using a new Kosmos-2 caption
query_caption = "The image shows an urban area with a river running through it."
query_inputs = clip_processor(text=query_caption, return_tensors="pt", truncation=True, max_length=77)

with torch.no_grad():
    query_embedding = clip_model.get_text_features(**query_inputs)
query_embedding = query_embedding / query_embedding.norm(p=2, dim=-1, keepdim=True)
query_vec = query_embedding.cpu().numpy()

# ✅ Search FAISS
k = 2  # Find top 2 closest captions
distances, indices = index.search(query_vec, k)

# ✅ Retrieve closest captions
print("\n🔍 Closest Captions:")
for idx in indices[0]:
    print(f"- {metadata[idx]['caption']} (from {metadata[idx]['image']})")

In [ ]:
# Print all vectors stored in the FAISS index
for i in range(index.ntotal):
    vec = index.reconstruct(i)
    print(f"Vector {i}: {vec}")


In [ ]:
  all_vectors = index.reconstruct_n(0, index.ntotal)
  print(all_vectors)

In [ ]:
print(f"Total vectors in FAISS index: {index.ntotal}")



In [ ]:
!pip install faiss-cpu langchain openai
